# Решение оптимизационной задачи по хранению контейнеров

Задача будет максимально упрощена, и будет представлять собой учебный пример.

## Общие условия:

Имеются несколько складов для хранения контейнеров, каждый склад может вместить ограниченное количество. Стоимсть хранения на складе разная, стоимость перемещения > 0, но при этом одинаковая для любого перемещения.

- Локации: Москва, Новомосковск, Калуга
- Тип контейнеров 1 40ft
- Стоимость перемещения 40 т.р.
- Перемещать можно в любую локацию из любой
- Стоимость хранения на локации т.р. в месяц:
    - Москва: 80
    - Новомосковск: 60
    - Калуга: 50
- Вместительность контейнерных складов:
    - Москва: 5000
    - Новомосковск: 300
    - Калуга: 400
- Оптимизируемые параметр: минимизация затрат в пересчете на месц.

## Иструмент для решения

OR-Tools



# Формализация задачи оптимизации хранения контейнеров

## 1. Назначение задачи
Определить оптимальное распределение контейнеров типа **40ft** по складам, минимизирующее суммарные **ежемесячные затраты**:
- хранение контейнеров,
- перемещение контейнеров между локациями.

Задача статическая, один расчётный месяц.

---

## 2. Локации
Множество складов:

$$
L = \{Москва,\ Новомосковск,\ Калуга\}
$$

Индекс локации: $i \in L$

---

## 3. Параметры

### 3.1 Стоимость хранения  
(тыс. руб. за контейнер в месяц)

- Москва: $c_{Москва} = 80$
- Новомосковск: $c_{Новомосковск} = 60$
- Калуга: $c_{Калуга} = 50$

---

### 3.2 Вместимость складов  
(контейнеров)

- Москва: $K_{Москва} = 5000$
- Новомосковск: $K_{Новомосковск} = 300$
- Калуга: $K_{Калуга} = 400$

---

### 3.3 Стоимость перемещения

Стоимость перемещения одного контейнера между любыми различными локациями:

$$
c_{move} = 40
$$

(тыс. руб.)

---

### 3.4 Общее количество контейнеров

$$
N
$$

задаётся извне.

---

## 4. Переменные решения

### 4.1 Хранение контейнеров

Количество контейнеров на складе $i$:

$$
x_i \ge 0,\quad x_i \in \mathbb{Z}
$$

---

### 4.2 Перемещения контейнеров

Количество контейнеров, перемещённых из локации $i$ в локацию $j$ за месяц:

$$
y_{ij} \ge 0,\quad y_{ij} \in \mathbb{Z},\quad i \ne j
$$

---

## 5. Ограничения

### 5.1 Баланс контейнеров

Суммарное количество контейнеров сохраняется:

$$
\sum_{i \in L} x_i = N
$$

---

### 5.2 Вместимость складов

Для каждой локации:

$$
x_i \le K_i
$$

---

### 5.3 Баланс с учётом перемещений

Для каждой локации $i$:

$$
x_i = x_i^{0} + \sum_{j \in L,\ j \ne i} y_{ji}
      - \sum_{j \in L,\ j \ne i} y_{ij}
$$

где $x_i^{0}$ — начальное количество контейнеров в локации $i$.

---

## 6. Целевая функция

Минимизировать суммарные ежемесячные затраты:

$$
\min Z =
\sum_{i \in L} c_i \cdot x_i
+
\sum_{i \in L} \sum_{j \in L,\ j \ne i} c_{move} \cdot y_{ij}
$$

---

## 7. Тип задачи

- линейная целочисленная оптимизация (ILP)
- целочисленные переменные
- решатель: **OR-Tools (CBC / SCIP)**

---

## 8. Упрощения модели

- один тип контейнеров;
- одинаковая стоимость перемещения;
- один временной период;
- отсутствуют ограничения на маршруты.

---

## 9. Назначение модели

Учебный пример для:
- формализации оптимизационной задачи;
- последующей реализации в OR-Tools.


In [28]:
# Установка OR-Tools (выполнить один раз)
!pip install ortools -q

In [29]:
from ortools.linear_solver import pywraplp
import pandas as pd

In [30]:
# Локации
locations = ['Москва', 'Новомосковск', 'Калуга']

# Стоимость хранения (тыс. руб. за контейнер в месяц)
storage_cost = {
    'Москва': 280,
    'Новомосковск': 60,
    'Калуга': 50
}

# Вместимость складов (контейнеров)
capacity = {
    'Москва': 5000,
    'Новомосковск': 300,
    'Калуга': 400
}

# Стоимость перемещения (тыс. руб. за контейнер)
move_cost = 40

# Начальное распределение контейнеров
initial_distribution = {
    'Москва': 400,
    'Новомосковск': 100,
    'Калуга': 300
}

# Общее количество контейнеров
N = sum(initial_distribution.values())
print(f"Общее количество контейнеров: {N}")

Общее количество контейнеров: 800


In [31]:
# Создаём солвер CBC (целочисленное программирование)
solver = pywraplp.Solver.CreateSolver('CBC')

if not solver:
    raise Exception("Солвер не создан!")
    
print("Солвер успешно создан")

Солвер успешно создан


In [32]:
# Переменные x_i — количество контейнеров на складе i (после перемещений)
x = {}
for loc in locations:
    x[loc] = solver.IntVar(0, capacity[loc], f'x_{loc}')

# Переменные y_ij — перемещение из i в j
y = {}
for i in locations:
    for j in locations:
        if i != j:
            # Максимум можно переместить все контейнеры из начальной локации
            max_move = initial_distribution[i]
            y[(i, j)] = solver.IntVar(0, max_move, f'y_{i}_{j}')

print(f"Создано переменных хранения: {len(x)}")
print(f"Создано переменных перемещения: {len(y)}")

Создано переменных хранения: 3
Создано переменных перемещения: 6


In [33]:
# Ограничение 1: Общий баланс контейнеров
solver.Add(sum(x[loc] for loc in locations) == N)
print("Добавлено: ограничение общего баланса")

# Ограничение 2: Вместимость складов (уже учтено в верхней границе переменных)
# Но добавим явно для наглядности
for loc in locations:
    solver.Add(x[loc] <= capacity[loc])
print("Добавлено: ограничения вместимости")

# Ограничение 3: Баланс с учётом перемещений для каждой локации
for loc in locations:
    # Входящие перемещения
    incoming = sum(y[(j, loc)] for j in locations if j != loc)
    # Исходящие перемещения
    outgoing = sum(y[(loc, j)] for j in locations if j != loc)
    # Баланс: конечное = начальное + входящие - исходящие
    solver.Add(x[loc] == initial_distribution[loc] + incoming - outgoing)

print("Добавлено: ограничения баланса перемещений")
print(f"Всего ограничений: {solver.NumConstraints()}")

Добавлено: ограничение общего баланса
Добавлено: ограничения вместимости
Добавлено: ограничения баланса перемещений
Всего ограничений: 7


In [34]:
# Затраты на хранение
storage_costs = sum(storage_cost[loc] * x[loc] for loc in locations)

# Затраты на перемещение
movement_costs = sum(move_cost * y[(i, j)] 
                     for i in locations 
                     for j in locations 
                     if i != j)

# Общие затраты — минимизируем
objective = storage_costs + movement_costs
solver.Minimize(objective)

print("Целевая функция задана: минимизация затрат")

Целевая функция задана: минимизация затрат


In [35]:
# Запуск решателя
status = solver.Solve()

# Проверка статуса
status_names = {
    pywraplp.Solver.OPTIMAL: "OPTIMAL",
    pywraplp.Solver.FEASIBLE: "FEASIBLE", 
    pywraplp.Solver.INFEASIBLE: "INFEASIBLE",
    pywraplp.Solver.UNBOUNDED: "UNBOUNDED",
    pywraplp.Solver.NOT_SOLVED: "NOT_SOLVED"
}

print(f"Статус решения: {status_names.get(status, 'UNKNOWN')}")

Статус решения: OPTIMAL


In [36]:
if status == pywraplp.Solver.OPTIMAL:
    print("=" * 60)
    print("ОПТИМАЛЬНОЕ РЕШЕНИЕ НАЙДЕНО")
    print("=" * 60)
    
    # Таблица распределения
    results = []
    for loc in locations:
        results.append({
            'Локация': loc,
            'Начальное кол-во': initial_distribution[loc],
            'Оптимальное кол-во': int(x[loc].solution_value()),
            'Изменение': int(x[loc].solution_value()) - initial_distribution[loc],
            'Стоимость хранения': storage_cost[loc],
            'Затраты (тыс.руб.)': int(x[loc].solution_value()) * storage_cost[loc]
        })
    
    df_results = pd.DataFrame(results)
    print("\n📦 РАСПРЕДЕЛЕНИЕ КОНТЕЙНЕРОВ:")
    print(df_results.to_string(index=False))
else:
    print("Оптимальное решение не найдено!")

ОПТИМАЛЬНОЕ РЕШЕНИЕ НАЙДЕНО

📦 РАСПРЕДЕЛЕНИЕ КОНТЕЙНЕРОВ:
     Локация  Начальное кол-во  Оптимальное кол-во  Изменение  Стоимость хранения  Затраты (тыс.руб.)
      Москва               400                 100       -300                 280               28000
Новомосковск               100                 300        200                  60               18000
      Калуга               300                 400        100                  50               20000


In [37]:
if status == pywraplp.Solver.OPTIMAL:
    print("\n🚚 ПЕРЕМЕЩЕНИЯ КОНТЕЙНЕРОВ:")
    
    movements = []
    total_moved = 0
    
    for i in locations:
        for j in locations:
            if i != j:
                qty = int(y[(i, j)].solution_value())
                if qty > 0:
                    movements.append({
                        'Откуда': i,
                        'Куда': j,
                        'Количество': qty,
                        'Стоимость (тыс.руб.)': qty * move_cost
                    })
                    total_moved += qty
    
    if movements:
        df_movements = pd.DataFrame(movements)
        print(df_movements.to_string(index=False))
        print(f"\nВсего перемещено контейнеров: {total_moved}")
    else:
        print("Перемещения не требуются")


🚚 ПЕРЕМЕЩЕНИЯ КОНТЕЙНЕРОВ:
Откуда         Куда  Количество  Стоимость (тыс.руб.)
Москва Новомосковск         200                  8000
Москва       Калуга         100                  4000

Всего перемещено контейнеров: 300


In [38]:
if status == pywraplp.Solver.OPTIMAL:
    print("\n" + "=" * 60)
    print("💰 ИТОГОВАЯ СВОДКА ЗАТРАТ")
    print("=" * 60)
    
    # Затраты на хранение
    total_storage = sum(int(x[loc].solution_value()) * storage_cost[loc] 
                        for loc in locations)
    
    # Затраты на перемещение
    total_movement = sum(int(y[(i, j)].solution_value()) * move_cost 
                         for i in locations 
                         for j in locations 
                         if i != j)
    
    # Затраты без оптимизации (всё хранить как есть)
    baseline_cost = sum(initial_distribution[loc] * storage_cost[loc] 
                        for loc in locations)
    
    print(f"Затраты на хранение:     {total_storage:,} тыс. руб.")
    print(f"Затраты на перемещение:  {total_movement:,} тыс. руб.")
    print(f"ИТОГО:                   {total_storage + total_movement:,} тыс. руб.")
    print("-" * 60)
    print(f"Затраты без оптимизации: {baseline_cost:,} тыс. руб.")
    print(f"Экономия:                {baseline_cost - (total_storage + total_movement):,} тыс. руб.")


💰 ИТОГОВАЯ СВОДКА ЗАТРАТ
Затраты на хранение:     66,000 тыс. руб.
Затраты на перемещение:  12,000 тыс. руб.
ИТОГО:                   78,000 тыс. руб.
------------------------------------------------------------
Затраты без оптимизации: 133,000 тыс. руб.
Экономия:                55,000 тыс. руб.


In [39]:
if status == pywraplp.Solver.OPTIMAL:
    print("\n" + "=" * 60)
    print("✅ ПРОВЕРКА КОРРЕКТНОСТИ РЕШЕНИЯ")
    print("=" * 60)
    
    # Проверка общего количества
    total_final = sum(int(x[loc].solution_value()) for loc in locations)
    print(f"Общее количество контейнеров: {total_final} (должно быть {N})")
    
    # Проверка вместимости
    print("\nПроверка вместимости:")
    for loc in locations:
        qty = int(x[loc].solution_value())
        cap = capacity[loc]
        status_icon = "✅" if qty <= cap else "❌"
        print(f"  {loc}: {qty}/{cap} {status_icon}")
    
    # Проверка баланса
    print("\nПроверка баланса перемещений:")
    for loc in locations:
        incoming = sum(int(y[(j, loc)].solution_value()) 
                      for j in locations if j != loc)
        outgoing = sum(int(y[(loc, j)].solution_value()) 
                      for j in locations if j != loc)
        expected = initial_distribution[loc] + incoming - outgoing
        actual = int(x[loc].solution_value())
        status_icon = "✅" if expected == actual else "❌"
        print(f"  {loc}: {initial_distribution[loc]} + {incoming} - {outgoing} = {expected} (факт: {actual}) {status_icon}")


✅ ПРОВЕРКА КОРРЕКТНОСТИ РЕШЕНИЯ
Общее количество контейнеров: 800 (должно быть 800)

Проверка вместимости:
  Москва: 100/5000 ✅
  Новомосковск: 300/300 ✅
  Калуга: 400/400 ✅

Проверка баланса перемещений:
  Москва: 400 + 0 - 300 = 100 (факт: 100) ✅
  Новомосковск: 100 + 200 - 0 = 300 (факт: 300) ✅
  Калуга: 300 + 100 - 0 = 400 (факт: 400) ✅
